# Spotify Audio Features — Data Analysis Project
### LDSCI4210 Intermediate Programming with Data | AE3

---

## Dataset Overview

The dataset used in this project is the **Spotify Song Attributes** dataset, sourced from Kaggle (publicly available, CC0: Public Domain licence). It contains **2,017 tracks** alongside a binary `target` column — whether the track was **liked (1)** or **disliked (0)** by a Spotify user — and **14 audio features** extracted by Spotify's own audio analysis API.

These features measure different sonic characteristics of each track:

| Feature | What it Measures |
|---|---|
| `danceability` | How suitable the track is for dancing (rhythm, beat stability) |
| `energy` | Perceived intensity and activity of the track |
| `valence` | Musical positiveness — high valence sounds happy, low valence sounds sad |
| `acousticness` | Confidence that the track is acoustic (not electronically produced) |
| `instrumentalness` | Probability that the track contains no vocals |
| `liveness` | Presence of a live audience in the recording |
| `speechiness` | Amount of spoken word content in the track |
| `loudness` | Average volume in decibels (dB), typically between −60 and 0 |
| `tempo` | Speed of the track in beats per minute (BPM) |

---

## Research Questions

| # | Research Question (Plain Language) | Techniques Applied |
|---|---|---|
| **RQ1** | *Can a computer learn your music taste just by listening to how a song sounds?* | k-NN classification, StandardScaler, train/test split, confusion matrix, evaluation metrics, scenario comparison |
| **RQ2** | *Do the songs you like actually sound different from the ones you skip?* | Grouped bar charts, box plots, Pearson correlation with target |
| **RQ3** | *Can we draw a simple "mood map" of all 2,017 songs using just two axes?* | PCA, cumulative variance, 2D projection, feature loadings |
| **RQ4** | *Are happy songs always energetic and danceable — or do mood and energy tell different stories?* | Correlation heatmap, scatter plot, pairplot |

---
## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print('All imports successful \u2713')

---
## 2. Class Definition: `SpotifyAnalyser`

Following the same Object-Oriented design pattern used in AE2, we encapsulate the entire analysis pipeline inside a single class. The `__init__` method acts as a central storage room for all data, models, and results — ensuring that every method can access what it needs without passing data back and forth manually.

In [ ]:
class SpotifyAnalyser:
    def __init__(self, filepath):
        """
        Initialises the SpotifyAnalyser with all attributes needed
        across the four research question pipelines.
        """
        # --- Data Storage ---
        self.filepath        = filepath
        self.dataset         = None
        self.cleaned_dataset = None

        # The nine continuous audio features used in all analyses
        self.feature_cols = [
            'acousticness', 'danceability', 'energy', 'instrumentalness',
            'liveness', 'loudness', 'speechiness', 'tempo', 'valence'
        ]

        # --- RQ1: Classification ---
        self.X_train            = None
        self.X_test             = None
        self.y_train            = None
        self.y_test             = None
        self.scaler             = None
        self.knn_model          = None
        self.predictions        = None
        self.evaluation_metrics = {}
        self.confusion_matrices = {}
        self.timing_data        = {}

        # --- RQ3: PCA ---
        self.pca_scaler   = None
        self.pca_model    = None
        self.X_2d         = None
        self.loadings     = None
        self.n_components = None

        # --- RQ4: Correlations ---
        self.corr_matrix = None

    # =========================================================
    # SECTION A: Data Loading and Preprocessing
    # =========================================================

    def load_data(self):
        """
        Loads the dataset from the CSV file path provided at initialisation.
        """
        self.dataset = pd.read_csv(self.filepath)
        # Remove the auto-generated index column Kaggle adds
        self.dataset = self.dataset.drop(columns=['Unnamed: 0'], errors='ignore')
        print(f'Data successfully loaded: {self.dataset.shape[0]} rows x {self.dataset.shape[1]} columns')

    def clean_data(self):
        """
        Performs preprocessing on the raw dataset:
          (i)  Removes duplicate rows
          (ii) Checks for and handles missing values in feature/target columns
          (iii)Validates that all feature columns contain numeric data
          (iv) Reports the class balance of the target column
        """
        df = self.dataset.copy()

        # (i) Remove duplicate rows
        initial_count = len(df)
        df = df.drop_duplicates()
        print(f'Removed {initial_count - len(df)} duplicate rows.')

        # (ii) Handle missing values in feature and target columns
        missing = df[self.feature_cols + ['target']].isnull().sum().sum()
        if missing > 0:
            df = df.dropna(subset=self.feature_cols + ['target'])
            print(f'Removed rows with missing values. Remaining: {len(df)}')
        else:
            print('No missing values found in feature or target columns. \u2713')

        # (iii) Validate numeric data types for feature columns
        for col in self.feature_cols:
            if not pd.api.types.is_numeric_dtype(df[col]):
                print(f'  Warning: column "{col}" is not numeric and will be excluded.')
                self.feature_cols.remove(col)

        # (iv) Report class balance
        counts = df['target'].value_counts()
        print(f'\nTarget distribution:')
        print(f'  Liked    (1): {counts[1]} tracks ({counts[1]/len(df)*100:.1f}%)')
        print(f'  Disliked (0): {counts[0]} tracks ({counts[0]/len(df)*100:.1f}%)')

        self.cleaned_dataset = df
        print('\nData cleaning complete. \u2713')

    # =========================================================
    # SECTION B: RQ1 — Taste Prediction via k-NN Classification
    # =========================================================

    def prepare_features(self, split_ratio=0.8):
        """
        Scales the audio features using StandardScaler and splits the data
        into training and testing sets.

        StandardScaler is essential here because k-NN uses Euclidean distance:
        without scaling, tempo (0-200 BPM) would dominate danceability (0-1).

        Parameters
        ----------
        split_ratio : float
            Proportion of data to use for training (default 0.8 = 80%).
        """
        X = self.cleaned_dataset[self.feature_cols]
        y = self.cleaned_dataset['target']

        # Standardise: each feature is transformed to mean=0, std=1
        self.scaler = StandardScaler()
        X_scaled    = self.scaler.fit_transform(X)

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X_scaled, y, train_size=split_ratio, random_state=42
        )
        print(f'Features scaled \u2713 | Train: {self.X_train.shape[0]} samples | Test: {self.X_test.shape[0]} samples')

    def train_model(self, k=5):
        """
        Trains the k-Nearest Neighbours classifier on the training set.

        Parameters
        ----------
        k : int — Number of neighbours to consider when voting (default 5).
        """
        self.knn_model = KNeighborsClassifier(n_neighbors=k)
        self.knn_model.fit(self.X_train, self.y_train)
        print(f'k-NN model trained successfully (k={k}). \u2713')

    def test_model(self):
        """
        Generates predictions on the test set using the trained k-NN model.
        """
        self.predictions = self.knn_model.predict(self.X_test)
        print('Predictions generated for test set. \u2713')
        return self.predictions

    def store_metrics(self, predictions, scenario_name):
        """
        Calculates and stores evaluation metrics and the confusion matrix
        for a given scenario.

        Parameters
        ----------
        predictions   : array-like — model predictions for this scenario
        scenario_name : str        — label for this scenario in results dictionaries
        """
        self.evaluation_metrics[scenario_name] = {
            'Accuracy' : accuracy_score(self.y_test, predictions),
            'Precision': precision_score(self.y_test, predictions, zero_division=0),
            'Recall'   : recall_score(self.y_test, predictions, zero_division=0),
            'F1-Score' : f1_score(self.y_test, predictions, zero_division=0)
        }
        self.confusion_matrices[scenario_name] = confusion_matrix(self.y_test, predictions)

    def visualise_confusion_matrix(self, predictions):
        """
        Plots a Seaborn heatmap of the confusion matrix.
        Shows how many songs were correctly and incorrectly classified
        as liked or disliked.
        """
        cm = confusion_matrix(self.y_test, predictions)

        plt.figure(figsize=(6, 5))
        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=['Disliked', 'Liked'],
            yticklabels=['Disliked', 'Liked']
        )
        plt.title('Confusion Matrix: k-NN Song Taste Prediction (k=5)', fontweight='bold')
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.tight_layout()
        plt.show()

    def find_optimal_k(self, max_k=20):
        """
        Tests k values from 1 to max_k and plots accuracy for each.
        Helps identify the best number of neighbours before running
        the full scenario comparison.

        Parameters
        ----------
        max_k : int — Upper bound of the k range to test (default 20).
        """
        k_range      = range(1, max_k + 1)
        acc_scores   = []

        for k in k_range:
            knn_k = KNeighborsClassifier(n_neighbors=k)
            knn_k.fit(self.X_train, self.y_train)
            acc_scores.append(accuracy_score(self.y_test, knn_k.predict(self.X_test)))

        best_k   = list(k_range)[acc_scores.index(max(acc_scores))]
        best_acc = max(acc_scores)

        plt.figure(figsize=(10, 5))
        plt.plot(k_range, acc_scores, marker='o', color='steelblue',
                 linewidth=2, markersize=6, label='Accuracy per k')
        plt.axvline(x=best_k, color='tomato', linestyle='--', linewidth=1.5,
                    label=f'Best k = {best_k}  (Accuracy = {best_acc:.3f})')
        plt.title('k-NN Accuracy vs. Number of Neighbours (k)', fontweight='bold')
        plt.xlabel('Number of Neighbours (k)')
        plt.ylabel('Accuracy')
        plt.xticks(list(k_range))
        plt.ylim(0.5, 1.0)
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

        print(f'Best k: {best_k} | Best Accuracy: {best_acc:.4f}')
        return best_k

    def compare_scenarios(self):
        """
        Runs the k-NN classification pipeline across four scenarios by
        varying both the train/test split ratio and the value of k.
        Records accuracy metrics and training/testing time for each,
        enabling a structured side-by-side comparison.
        """
        scenarios = [
            {'split': 0.8, 'k': 5, 'name': '80-20 split, k=5'},
            {'split': 0.7, 'k': 5, 'name': '70-30 split, k=5'},
            {'split': 0.8, 'k': 7, 'name': '80-20 split, k=7'},
            {'split': 0.7, 'k': 7, 'name': '70-30 split, k=7'},
        ]

        for s in scenarios:
            print(f'\nRunning: {s["name"]}')

            # Re-prepare features with the specified split ratio
            self.prepare_features(split_ratio=s['split'])

            # Train and record time
            start_train = time.time()
            self.train_model(k=s['k'])
            end_train   = time.time()

            # Test and record time
            start_test  = time.time()
            preds       = self.test_model()
            end_test    = time.time()

            # Store timing data
            train_time = end_train - start_train
            test_time  = end_test  - start_test
            self.timing_data[s['name']] = {
                'Training Time (s)': round(train_time, 4),
                'Testing Time (s)' : round(test_time,  4),
                'Total Time (s)'   : round(train_time + test_time, 4)
            }

            # Store evaluation metrics and confusion matrix
            self.store_metrics(preds, s['name'])

    def plot_matrix_grid(self):
        """
        Displays a 2x2 grid of confusion matrices, one per scenario,
        so that classification outcomes can be compared at a glance.
        """
        fig, axes = plt.subplots(2, 2, figsize=(11, 8))
        axes = axes.flatten()

        for idx, (scenario, cm) in enumerate(self.confusion_matrices.items()):
            disp = ConfusionMatrixDisplay(
                confusion_matrix=cm,
                display_labels=['Disliked', 'Liked']
            )
            disp.plot(ax=axes[idx], cmap='Blues', colorbar=False)
            axes[idx].set_title(scenario, fontweight='bold')

        plt.suptitle('Confusion Matrix Comparison Across All Scenarios',
                     fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

    def plot_accuracy_chart(self):
        """
        Generates a grouped bar chart comparing all four evaluation metrics
        (Accuracy, Precision, Recall, F1-Score) across all scenarios.
        """
        metrics_df = pd.DataFrame(self.evaluation_metrics).T

        metrics_df.plot(
            kind='bar',
            figsize=(11, 6),
            colormap='Set2',
            edgecolor='white',
            width=0.75
        )
        plt.title('Classification Metrics Across All Scenarios', fontweight='bold')
        plt.ylabel('Score (0 to 1)')
        plt.ylim(0, 1.0)
        plt.xticks(rotation=15, ha='right')
        plt.legend(loc='lower right')
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

    # =========================================================
    # SECTION C: RQ2 — Do Liked Songs Sound Different?
    # =========================================================

    def compare_feature_groups(self):
        """
        Computes and prints the mean value of each audio feature
        for liked and disliked songs, and calculates the Pearson
        correlation of each feature with the target label.
        Mirrors the group-comparison methodology from AE1.
        """
        df  = self.cleaned_dataset
        avg = df.groupby('target')[self.feature_cols].mean()

        print('--- Average Feature Values by Group ---')
        print(avg.T.round(3).to_string())

        corr = (df[self.feature_cols + ['target']]
                .corr()['target']
                .drop('target')
                .sort_values(ascending=False))

        print('\n--- Pearson Correlation with Target (Liked = 1) ---')
        print(corr.round(4).to_string())

        return avg, corr

    def visualise_feature_boxplots(self):
        """
        Plots a 2x3 grid of side-by-side box plots for the six most
        perceptually meaningful features, split by liked vs disliked.
        Box plots show the median, spread, and outliers for each group.
        """
        df    = self.cleaned_dataset
        liked = df[df['target'] == 1]
        dis   = df[df['target'] == 0]

        key_features = ['danceability', 'energy', 'valence',
                        'acousticness', 'loudness', 'tempo']

        fig, axes = plt.subplots(2, 3, figsize=(14, 8))
        axes = axes.flatten()

        for i, feat in enumerate(key_features):
            bp = axes[i].boxplot(
                [dis[feat], liked[feat]],
                labels=['Disliked', 'Liked'],
                patch_artist=True,
                medianprops=dict(color='black', linewidth=2)
            )
            bp['boxes'][0].set_facecolor('#E07070')
            bp['boxes'][1].set_facecolor('#70B0E0')
            axes[i].set_title(feat.capitalize(), fontweight='bold')
            axes[i].set_ylabel(feat)
            axes[i].grid(True, linestyle='--', alpha=0.5)

        plt.suptitle('Do Liked Songs Actually Sound Different? Feature-by-Feature Comparison',
                     fontsize=13, fontweight='bold', y=1.01)
        plt.tight_layout()
        plt.show()

    def visualise_feature_barchart(self):
        """
        Plots a normalised grouped bar chart comparing average feature values
        between liked and disliked songs. Features are min-max normalised
        so that all nine can be compared on the same 0-1 axis regardless
        of their original scale.
        """
        df  = self.cleaned_dataset
        avg = df.groupby('target')[self.feature_cols].mean()

        # Min-max normalise using the full dataset range
        avg_norm = avg.copy()
        for col in self.feature_cols:
            col_min = df[col].min()
            col_max = df[col].max()
            avg_norm[col] = (avg[col] - col_min) / (col_max - col_min)

        plot_data = avg_norm.T
        plot_data.columns = ['Disliked (0)', 'Liked (1)']

        plot_data.plot(
            kind='bar',
            figsize=(13, 6),
            color=['#E07070', '#70B0E0'],
            edgecolor='white',
            width=0.7
        )
        plt.title('Normalised Average Audio Features: Liked vs Disliked Songs\n'
                  '(All features rescaled to 0-1 for a fair comparison)',
                  fontweight='bold')
        plt.xlabel('Audio Feature')
        plt.ylabel('Normalised Average Value')
        plt.xticks(rotation=35, ha='right')
        plt.legend(fontsize=11)
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

    def visualise_correlation_bar(self):
        """
        Plots a horizontal bar chart of each feature's Pearson correlation
        with the target. Blue bars indicate features more common in liked
        songs; red bars indicate the opposite.
        """
        df   = self.cleaned_dataset
        corr = (df[self.feature_cols + ['target']]
                .corr()['target']
                .drop('target')
                .sort_values())

        colours = ['#70B0E0' if v > 0 else '#E07070' for v in corr]

        plt.figure(figsize=(10, 5))
        plt.barh(corr.index, corr.values, color=colours, edgecolor='white')
        plt.axvline(0, color='black', linewidth=0.8)
        plt.title('Which Sound Features Are More Common in Liked Songs?\n'
                  '(Blue = more present in liked songs | Red = more present in disliked songs)',
                  fontweight='bold')
        plt.xlabel('Correlation with Liked (target = 1)')
        plt.grid(axis='x', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

    # =========================================================
    # SECTION D: RQ3 — Building a Mood Map with PCA
    # =========================================================

    def apply_pca(self, target_variance=0.80):
        """
        Applies PCA to the audio feature matrix to reduce dimensionality.
        First performs exploratory PCA to find how many components are
        needed to explain target_variance of the data, then reduces to
        2D for visualisation.

        Parameters
        ----------
        target_variance : float
            Proportion of total variance to capture (default 0.80 = 80%).
        """
        df = self.cleaned_dataset

        # Scale the features (PCA is sensitive to scale)
        self.pca_scaler = StandardScaler()
        X_scaled        = self.pca_scaler.fit_transform(df[self.feature_cols])

        # Exploratory PCA: how many components reach the target variance?
        pca_explore        = PCA(n_components=len(self.feature_cols), random_state=42)
        pca_explore.fit(X_scaled)
        cumulative_variance = np.cumsum(pca_explore.explained_variance_ratio_)

        self.n_components = int(np.argmax(cumulative_variance >= target_variance)) + 1

        # Plot cumulative variance curve
        plt.figure(figsize=(9, 5))
        plt.plot(
            range(1, len(self.feature_cols) + 1), cumulative_variance,
            marker='o', markersize=6, color='seagreen', linewidth=2,
            label='Cumulative Explained Variance'
        )
        plt.axhline(y=target_variance, color='tomato', linestyle='--', linewidth=1.5,
                    label=f'{int(target_variance*100)}% Variance Threshold')
        plt.axvline(x=self.n_components, color='darkorange', linestyle=':', linewidth=1.5,
                    label=f'{self.n_components} components needed')
        plt.title('How Many "Mood Axes" Do We Actually Need?\n'
                  'Cumulative Explained Variance by Number of PCA Components',
                  fontweight='bold')
        plt.xlabel('Number of Components')
        plt.ylabel('Cumulative Explained Variance')
        plt.xticks(range(1, len(self.feature_cols) + 1))
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

        print(f'Components needed to capture {int(target_variance*100)}% of variance: {self.n_components}')

        # Final 2D PCA for visualisation
        self.pca_model = PCA(n_components=2, random_state=42)
        self.X_2d      = self.pca_model.fit_transform(X_scaled)

        # Store feature loadings for interpretation in the next step
        self.loadings = pd.DataFrame(
            self.pca_model.components_.T,
            index=self.feature_cols,
            columns=['PC1', 'PC2']
        )
        print('PCA complete. \u2713')

    def visualise_pca_projection(self):
        """
        Plots all 2,017 songs on a 2D mood map where each axis
        represents one principal component. Songs are coloured by
        whether they were liked or disliked.
        """
        df      = self.cleaned_dataset
        pc1_var = self.pca_model.explained_variance_ratio_[0] * 100
        pc2_var = self.pca_model.explained_variance_ratio_[1] * 100

        fig, ax = plt.subplots(figsize=(10, 7))

        for label, colour, name in [(0, '#E07070', 'Disliked'), (1, '#70B0E0', 'Liked')]:
            mask = df['target'].values == label
            ax.scatter(
                self.X_2d[mask, 0], self.X_2d[mask, 1],
                c=colour, alpha=0.45, s=25,
                edgecolors='none', label=name
            )

        ax.set_title('A Mood Map of 2,017 Songs\u2014Can You Spot a Pattern?\n'
                     'Each dot is one track, plotted by its two dominant sound dimensions',
                     fontweight='bold')
        ax.set_xlabel(f'PC1 \u2014 Intensity Axis ({pc1_var:.1f}% of variance)')
        ax.set_ylabel(f'PC2 \u2014 Mood Positivity Axis ({pc2_var:.1f}% of variance)')
        ax.legend(fontsize=11)
        ax.grid(True, linestyle='--', alpha=0.3)
        plt.tight_layout()
        plt.show()

    def visualise_pca_loadings(self):
        """
        Plots a grouped bar chart of PCA feature loadings.
        Reveals which original audio features drive each principal
        component, making the abstract axes interpretable in plain terms.
        """
        self.loadings.plot(
            kind='bar',
            figsize=(11, 5),
            color=['steelblue', 'coral'],
            edgecolor='white',
            width=0.7
        )
        plt.axhline(0, color='black', linewidth=0.8)
        plt.title('What Does Each Mood Axis Actually Measure?\n'
                  'PCA Feature Loadings: Contribution of Each Feature to PC1 and PC2',
                  fontweight='bold')
        plt.ylabel('Loading Weight')
        plt.xlabel('Audio Feature')
        plt.xticks(rotation=30, ha='right')
        plt.legend(fontsize=11)
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

        print('\n--- Feature Loadings Table ---')
        print(self.loadings.round(3).to_string())

    # =========================================================
    # SECTION E: RQ4 — How Do Mood Dimensions Connect?
    # =========================================================

    def analyse_mood_correlations(self):
        """
        Computes the Pearson correlation matrix across all audio features
        and identifies the strongest positive and negative feature pairs.
        """
        df               = self.cleaned_dataset
        self.corr_matrix = df[self.feature_cols].corr()

        # Extract all unique upper-triangle feature pairs
        pairs = (
            self.corr_matrix
            .where(np.triu(np.ones(self.corr_matrix.shape), k=1).astype(bool))
            .stack().reset_index()
        )
        pairs.columns = ['Feature A', 'Feature B', 'Pearson r']

        print('--- Top 5 Most Positively Correlated Feature Pairs ---')
        print(pairs.nlargest(5, 'Pearson r').to_string(index=False))

        print('\n--- Top 5 Most Negatively Correlated Feature Pairs ---')
        print(pairs.nsmallest(5, 'Pearson r').to_string(index=False))

    def visualise_heatmap(self):
        """
        Plots a Seaborn correlation heatmap of all audio features.
        Warm colours (red) indicate a strong positive relationship;
        cool colours (blue) indicate a strong negative one.
        """
        plt.figure(figsize=(11, 8))
        sns.heatmap(
            self.corr_matrix,
            annot=True,
            fmt='.2f',
            cmap='coolwarm',
            center=0,
            linewidths=0.5,
            square=True,
            cbar_kws={'label': 'Pearson r'}
        )
        plt.title("How Do a Song's Sound Dimensions Relate to Each Other?\n"
                  'Correlation Heatmap of All Audio Features',
                  fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

    def visualise_mood_scatter(self):
        """
        Plots Energy vs Danceability for all songs, with each point
        coloured by its Valence score. This encodes three mood dimensions
        simultaneously: position (energy/dance) and colour (happiness).
        """
        df = self.cleaned_dataset

        plt.figure(figsize=(10, 7))
        scatter = plt.scatter(
            df['energy'],
            df['danceability'],
            c=df['valence'],
            cmap='RdYlGn',
            alpha=0.55,
            edgecolors='none',
            s=30
        )
        cbar = plt.colorbar(scatter)
        cbar.set_label('Valence  (0 = Sad/Negative  \u2192  1 = Happy/Positive)', fontsize=10)
        plt.title('Are Happy Songs Always Energetic and Danceable?\n'
                  'Energy vs Danceability, Coloured by Happiness (Valence)',
                  fontweight='bold')
        plt.xlabel('Energy  (0 = Calm  \u2192  1 = Intense)')
        plt.ylabel('Danceability  (0 = Hard to dance to  \u2192  1 = Very danceable)')
        plt.grid(True, linestyle='--', alpha=0.3)
        plt.tight_layout()
        plt.show()

    def visualise_pairplot(self):
        """
        Generates a Seaborn pairplot of the four most musically intuitive
        features, split by liked vs disliked. The diagonal shows the KDE
        distribution for each group; off-diagonal cells show scatter plots
        for each feature pair.
        """
        df = self.cleaned_dataset.copy()
        df['Preference'] = df['target'].map({0: 'Disliked', 1: 'Liked'})

        pair_grid = sns.pairplot(
            df[['danceability', 'energy', 'valence', 'acousticness', 'Preference']],
            hue='Preference',
            palette={'Disliked': '#E07070', 'Liked': '#70B0E0'},
            plot_kws={'alpha': 0.35, 's': 15},
            diag_kind='kde'
        )
        pair_grid.figure.suptitle(
            'How Do the Key Mood Features Relate to Each Other?\n'
            'Pairwise Comparison: Danceability, Energy, Valence & Acousticness',
            y=1.02, fontsize=12, fontweight='bold'
        )
        plt.tight_layout()
        plt.show()


print('SpotifyAnalyser class defined successfully \u2713')

---
## 3. Data Loading and Preprocessing

In [ ]:
spotify = SpotifyAnalyser('data.csv')
spotify.load_data()
spotify.clean_data()

In [ ]:
# Preview the cleaned dataset
spotify.cleaned_dataset.head()

In [ ]:
# Summary statistics for all audio features
spotify.cleaned_dataset[spotify.feature_cols].describe().round(3)

---
## RQ1 — *Can a computer learn your music taste just by listening to how a song sounds?*

Most people know within a few seconds whether they like a song. But how much of that gut reaction is actually encoded in measurable sound properties — how fast the beat is, how intense the track feels, how easy it is to dance to?

To answer this, we train a **k-Nearest Neighbours (k-NN)** classifier. The idea is intuitive: if two songs sound similar, they are probably both liked or both disliked. k-NN finds the *k* most sonically similar songs to a new track and takes a majority vote on whether it should be liked.

Before training, all features are **standardised** — this ensures the algorithm treats a 10-point difference in tempo the same way it treats a 0.1 difference in danceability, rather than letting large-scale features dominate the distance calculation.

In [ ]:
# Prepare features and split into 80% training / 20% testing
spotify.prepare_features(split_ratio=0.8)
spotify.train_model(k=5)
predictions = spotify.test_model()
spotify.visualise_confusion_matrix(predictions)

In [ ]:
# Find the optimal value of k before running the full scenario comparison
spotify.find_optimal_k(max_k=20)

In [ ]:
# Run all four scenarios: vary the split ratio and k
spotify.compare_scenarios()

In [ ]:
spotify.plot_matrix_grid()
spotify.plot_accuracy_chart()

### RQ1 — Findings

The k-NN classifier achieves an accuracy of roughly **70–73%** across all four scenarios, with the best result at an 80-20 split using k=7. This is meaningfully above the 50% baseline a random guesser would achieve, confirming that **audio features do carry real information about whether a song will be liked**.

The confusion matrices show balanced performance across both classes — the model is not simply defaulting to one label, which is reassuring given that the dataset is evenly balanced between liked and disliked tracks.

The optimal k plot shows accuracy peaking around k=7 before declining — a pattern driven by the **bias-variance trade-off**: very low k values memorise noise in the training data (overfitting), while very high values over-generalise and lose discriminative power.

However, a 70% accuracy ceiling also tells an important story: **sound alone does not fully explain musical preference**. Factors such as artist familiarity, genre expectations, and listening context — none of which are captured by audio features — account for a significant portion of what makes a song likeable to any particular listener.

---
## RQ2 — *Do the songs you like actually sound different from the ones you skip?*

Before asking a computer to predict taste, it is worth asking a simpler human question: **is there even a detectable sonic difference between liked and disliked songs?** A person who enjoys hip-hop might consistently favour tracks with a strong, rhythmic beat. A folk music fan might show the opposite pattern.

We compare the distribution of each audio feature between the two groups using **box plots** (to see the spread and median), a **normalised grouped bar chart** (to compare averages side-by-side), and a **correlation bar chart** (to quantify the direction and strength of each feature's relationship with being liked).

This mirrors the group-comparison methodology used in AE1, where depressive and non-depressive tweet groups were compared by their linguistic characteristics.

In [ ]:
avg_by_group, correlations = spotify.compare_feature_groups()

In [ ]:
spotify.visualise_feature_boxplots()

In [ ]:
spotify.visualise_feature_barchart()

In [ ]:
spotify.visualise_correlation_bar()

### RQ2 — Findings

Yes — liked and disliked songs do sound measurably different, though the differences are subtle rather than dramatic.

**Danceability** is the strongest positive differentiator (r = 0.18): the songs this user tends to like have a more consistent, rhythmic beat. **Speechiness** and **instrumentalness** also show a positive relationship with being liked, suggesting a preference for vocal-forward tracks or tracks with clear instrumental character.

On the opposite side, **acousticness** is the strongest negative predictor (r = −0.13): songs that sound organic or acoustic are more likely to be disliked. Combined with the danceability finding, the picture that emerges is of a listener who prefers **produced, beat-driven music** over acoustic tracks.

The box plots reveal considerable overlap between both groups across every feature — confirming that while the *average* differences are real, no single feature cleanly separates liked from disliked songs. This is precisely why the k-NN classifier in RQ1 achieved moderate rather than high accuracy: **taste is encoded across the combination of all features, not in any one of them alone**.

---
## RQ3 — *Can we draw a simple "mood map" of all 2,017 songs using just two axes?*

Spotify describes each song using nine different audio features — nine separate dimensions that are impossible to visualise all at once. **Principal Component Analysis (PCA)** compresses this complexity into a smaller set of summary axes called "principal components", each of which captures the most important patterns of variation across all the features simultaneously.

The goal is to reduce all nine features down to just two — enough to plot every song as a single dot on a 2D mood map — and then ask: do liked and disliked songs end up in different regions of this map? And what do the two axes actually represent musically?

This directly mirrors the dimensionality reduction step used in AE2, where PCA compressed thousands of TF-IDF word features before classification.

In [ ]:
# Apply PCA — this generates the cumulative variance plot automatically
spotify.apply_pca(target_variance=0.80)

In [ ]:
# Plot the 2D mood map of all 2,017 songs
spotify.visualise_pca_projection()

In [ ]:
# Interpret what each axis actually represents
spotify.visualise_pca_loadings()

### RQ3 — Findings

The cumulative variance plot shows that **6 components are required to capture 80% of the variance** in the nine audio features. This means the feature space cannot be drastically compressed without meaningful loss — the nine features are measuring genuinely distinct aspects of a song's sound, not redundant versions of the same thing.

The 2D mood map plots all 2,017 songs on two axes. Liked and disliked songs **occupy largely the same region** — their distributions overlap heavily, reinforcing the finding from RQ1 that audio features alone do not perfectly separate taste.

The feature loadings chart reveals the musical interpretation of each axis:
- **PC1 (Intensity Axis):** Driven by energy and loudness at one end and acousticness at the other. Moving right on the map means louder, more intense, and more electronically produced. Moving left means quieter and acoustic.
- **PC2 (Mood-Positivity Axis):** Driven by valence and danceability at one end and instrumentalness at the other. Moving upward means happier and more danceable. Moving downward means more abstract or purely instrumental.

Together, these two axes form an interpretable sonic coordinate system: a song in the top-right is high-energy and happy (upbeat pop, electronic dance); a song in the bottom-left is quiet and instrumental (ambient, classical).

---
## RQ4 — *Are happy songs always energetic and danceable — or do mood and energy tell different stories?*

Intuitively, we might assume that a happy, upbeat song is also energetic and easy to dance to. But is that really true? Could a song score high on happiness while being calm and low-energy? Or are these so tightly bound they essentially measure the same thing?

We answer this by computing a **full correlation heatmap** across all audio features, then plotting **Energy vs Danceability coloured by Valence** to display three mood dimensions simultaneously, and finally using a **pairplot** to examine every pairwise combination across the four key mood features.

In [ ]:
spotify.analyse_mood_correlations()

In [ ]:
spotify.visualise_heatmap()

In [ ]:
spotify.visualise_mood_scatter()

In [ ]:
spotify.visualise_pairplot()

### RQ4 — Findings

The heatmap reveals two particularly strong relationships:

- **Energy and loudness** are strongly positively correlated (r ≈ 0.76). Louder tracks are consistently perceived as more intense and energetic — an expected psychoacoustic finding. These two features are largely measuring the same underlying property, which explains why six components were needed in RQ3 rather than fewer.

- **Acousticness and energy** are strongly negatively correlated (r ≈ −0.65). Produced, electronic music tends to be both energetic and loud, while acoustic music tends to be quiet and calm. This pair acts as the primary dividing axis in the dataset — consistent with the finding from RQ3 that PC1 was dominated by exactly these two features.

The scatter plot of energy vs danceability coloured by valence answers the core question directly: **happy songs are not always energetic, and energetic songs are not always happy.** The valence-energy correlation is weak (r ≈ 0.10) — mood and intensity are genuinely separate sonic dimensions.

However, **valence and danceability do show a clearer positive relationship**: tracks that sound musically positive also tend to be more rhythmically compelling. The pairplot reinforces this and shows that liked songs shift slightly but consistently toward higher danceability across all pairwise comparisons — connecting the findings from RQ2, RQ3, and RQ4 into a coherent picture of what drives this listener's preferences.

---
## Overall Conclusions

| Research Question | Answer |
|---|---|
| **RQ1** — Can sound predict taste? | Partially — k-NN achieves ~70–73% accuracy, meaningfully above chance but limited by the subjective nature of preference |
| **RQ2** — Do liked songs sound different? | Yes, subtly — danceability is higher in liked songs; acousticness is lower; but distributions overlap heavily across all features |
| **RQ3** — Can we build a 2D mood map? | Yes — PC1 is an intensity axis (energy/loudness vs acousticness); PC2 is a mood-positivity axis (valence/danceability vs instrumentalness); but liked and disliked songs overlap in this space |
| **RQ4** — Are happy songs always energetic? | No — valence and energy are weakly correlated; mood and intensity are independent sonic dimensions; valence correlates more strongly with danceability |

### Limitations
- The `target` column reflects **one individual's taste**, limiting the generalisability of the classification findings.
- Factors such as genre, artist recognition, release year, and listening context are absent from the dataset and likely explain much of the unexplained variance.
- Pearson correlation measures only **linear relationships** — non-linear dependencies between features may exist but are not captured here.
- With 2,017 tracks the dataset is appropriate for exploratory analysis but may be small for robust machine learning conclusions.

---
*Dataset: Spotify Song Attributes, Kaggle (CC0: Public Domain)*

---
## Reflection

### Object-Oriented Design Decisions and Benefits

I designed the `SpotifyAnalyser` class to act as a self-contained analysis pipeline, following the same architectural approach used in AE2's `MentalHealthClassifier`. The `__init__` method serves as a central storage room: every artefact produced during the analysis — the cleaned dataset, the trained model, PCA results, correlation matrices, evaluation metrics — is stored as an instance attribute, ensuring all methods can access shared state without duplicating data or passing large objects between functions.

The primary benefit of this approach is **extensibility without fragmentation**. When it came time to add PCA for RQ3, I did not need to touch the loading or preprocessing logic — I simply added `apply_pca()`, `visualise_pca_projection()`, and `visualise_pca_loadings()` to the class. This modularity also makes the execution cells at the bottom of the notebook very readable: `spotify.compare_scenarios()` runs an entire four-scenario classification pipeline in a single call, hiding the complexity behind a clean, human-readable interface — the same design benefit highlighted in AE2's reflection.

### Insights on Classification and Dimensionality Reduction

One of the most significant insights was the contrast between how PCA was used in AE2 versus here. In AE2, the TF-IDF feature matrix was sparse and extremely high-dimensional (thousands of columns), making PCA an essential pre-processing step to counter the *curse of dimensionality* before k-NN could perform reliably. Here, the feature space is far smaller (only nine features), so StandardScaler is sufficient for the classifier and PCA's role shifts entirely: it becomes a **visualisation and interpretation tool** rather than a classification aid.

The cumulative variance curve showing that six components are required to explain 80% of the variance was itself an interesting finding — it tells us that the nine audio features are not interchangeable or redundant, but are genuinely measuring distinct sonic properties. This is a direct consequence of having domain-specific engineered features (Spotify's API) rather than high-dimensional bag-of-words representations.

### Strengths, Limitations, and Implementation Challenges

A key strength of this implementation is the `compare_scenarios()` method, which automates metric storage and timing across four different configurations — mirroring the same design in AE2 — and makes adding new scenarios trivial without modifying any other part of the code. However, the same strict method-dependency limitation applies: methods must be called in the correct order (`load_data` → `clean_data` → `prepare_features` → `train_model`), because each method depends on instance attributes set by the previous one. Calling them out of sequence would raise an `AttributeError` because those attributes have not yet been populated.

A challenge specific to this project was the normalisation step inside `visualise_feature_barchart()`. Because audio features operate on entirely different scales — tempo spans roughly 60 to 200 BPM while danceability is bounded between 0 and 1 — plotting raw averages side-by-side would have been visually misleading. Applying min-max normalisation per feature resolves this, but it also means the chart communicates relative within-feature differences rather than absolute magnitudes. This trade-off must be stated clearly whenever the chart is presented.

### Responsible Real-World Application

A song preference classifier of this kind underpins real-world recommendation systems at scale. When deployed responsibly, such a model should always be framed as a **starting point for discovery**, not a definitive judgment of taste. Over-relying on past preference patterns creates *filter bubbles*, where users are only ever recommended music similar to what they have already heard — at the cost of exposure to new genres, artists, and cultural contexts that they may come to enjoy.

Additionally, because the `target` column reflects a single user's taste, any model trained on this data would require extensive retraining and cross-user validation before being generalised. Personalised recommendation at the scale of a streaming platform combines content-based audio features with collaborative filtering across millions of listeners — a complexity far beyond this project, but important context for interpreting these findings responsibly.